## 2. 벡터 저장소 (Vector Store)

## 2.1 Chroma 

In [4]:

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

from langchain_chroma import Chroma

chroma_db = Chroma(
    collection_name = "api_simple_collection",
    embedding_function = embeddings_model,
    persist_directory = "./chroma_db"
)

In [9]:
from langchain_elasticsearch import ElasticsearchStore

vector_poi = "vector_poi"

vector_store = ElasticsearchStore(
    embedding=embeddings_model,
    index_name=vector_poi,
    es_url="http://localhost:9200"
)

In [20]:
import pandas as pd

fielePath = "./data/poi_data.csv"

def read_excel_as_list(file_path):
    # Excel 파일 읽기
    try:
        # pandas로 Excel 파일 읽기
        df = pd.read_csv(file_path, encoding='EUC-KR')
        
        # DataFrame을 리스트로 변환
        data_list = df.values.tolist()

        return data_list

    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return []

data_list = read_excel_as_list(file_path=fielePath)
print(data_list[0])
print(len(data_list))



['A', '골목상권', 3110008, '배화여자대학교(박노수미술관)', 197093, 453418, 11110, '종로구', 11110515, '청운효자동', 149264]
1650


In [38]:
import math

a    = 6378137.0               
f    = 1 / 298.257222101      
lat0 = 38.0 * math.pi / 180.0  
lon0 = 127.0 * math.pi / 180.0 
k0   = 1.0                     
x0   = 200000.0                
y0   = 500000.0   

def tm_to_wgs84(x, y):

    math.sqrt(2 * f- f * f)
    n = f / (2 - f)
    A = a / (1 + n) * (1 + n*n/4 + n*n*n*n/64)

    x = x - x0
    y = y - y0

    lat = lat0 
    for i in range(5):
        lat = (y / (k0 * A)) + lat0
	
    lon = lon0 + (x / (k0 * A * math.cos(lat)))
    lat_deg = lat * 180.0 / math.pi
    lon_deg = lon * 180.0 / math.pi

    return lon_deg, lat_deg

# embedding field using embedded model 


In [92]:
# declare variance
es_url = "http://localhost:9200"
es_index_name = "vector_poi"

mapping = {
    "mappings": {
        "properties": {
            "title_vector": {  # title 벡터 필드
                "type": "dense_vector",
                "dims": 1024  # Hugging Face 모델에서 생성된 벡터 차원 수
            },
            "address_vector": {  # address 벡터 필드
                "type": "dense_vector",
                "dims": 1024
            },
            "location": {  # lat, lon을 포함하는 geo_point 필드
                "type": "geo_point"
            },
            "title": {  # title 원본 텍스트
                "type": "text"
            },
            "address": {  # address 원본 텍스트
                "type": "text"
            }
        }
    }
}

template = {
    "title": "{{title}}",
    "address": "{{address}}",
    "location": {
        "lat": "{{lat}}",
        "lon": "{{lon}}"
    },
    "vector": "{{vector}}"
}



In [93]:
from elasticsearch import Elasticsearch

# Elasticsearch 클라이언트 초기화
es = Elasticsearch("http://localhost:9200")


# 인덱스 생성
if not es.indices.exists(index=es_index_name):
    es.indices.create(index=es_index_name, body=mapping)
    print(f"Elasticsearch 인덱스 '{es_index_name}'가 생성되었습니다.")
else:
    print(f"Elasticsearch 인덱스 '{es_index_name}'는 이미 존재합니다.")

Elasticsearch 인덱스 'vector_poi'가 생성되었습니다.


In [58]:
bulk_data = []
for doc in data_list:
    lon, lat = tm_to_wgs84(doc[4], doc[5])
    print(doc[3], doc[9])
    title_vector = embeddings_model.embed_query(doc[3])
    address_vector = embeddings_model.embed_query(doc[9])
    source = {
         "_index": es_index_name, 
        "_source": {
            "title": doc[3],
            "address": doc[9],
            "title_vector": title_vector,
            "address_vector": address_vector,
            "location": {"lat": lat, "lon": lon}
        }
    }
    bulk_data.append(source)


    

배화여자대학교(박노수미술관) 청운효자동
자하문터널 부암동
평창동서측 평창동
정독도서관 가회동
중앙고등학교 가회동
창덕궁 종로1?2?3?4가동
서울국제고등학교 혜화동
이북5도청사 평창동
독립문역 1번 무악동
세검정초등학교 부암동
대신고등학교 교남동
동묘앞역 2번 숭인2동
청운초등학교 청운효자동
성곡미술관 사직동
체부동홍종문가옥 사직동
경복고등학교 청운효자동
청와대사랑채 청운효자동
평창동동측 평창동
세검정 부암동
부암동주민센터 부암동
사직공원(한국사회과학도서관) 사직동
신설동역 12번 숭인2동
경향신문사 소공동
회현역 1번 회현동
성균관대학교 혜화동
경신고등학교 혜화동
서울대병원 이화동
혜회동주민센터 혜화동
충신시장옆 종로5?6가동
종로5가역 4번 종로5?6가동
혜화역 1번 이화동
이화달팽이길 이화동
동대문역 1번 창신2동
동대문역 3번 창신1동
창신역 1번 창신3동
수족관거리 창신1동
창신역 4번 숭인1동
창신1동주민센터 창신1동
황학코아루아파트 황학동
성심여자고등학교 원효로2동
새남터성지 이촌2동
동묘앞역 3번 숭인2동
중구회현체육센터(정화예술대남산캠퍼스) 회현동
남산케이블카 명동
남산골공원옆 필동
충무초등학교 장충동
관성묘 장충동
동대문역사문화공원역 5번 장충동
버티고개 다산동
다산성곽길 다산동
장충동주민센터 장충동
장충단 고개 다산동
청구역 1번 신당동
청구역 3번 청구동
황학동벼룩시장 황학동
신당역 3번 신당5동
후암동주민센터 후암동
해방촌예술마을 후암동
해방촌 남동측 용산2가동
약수역 7번 약수동
한양공고앞 교차로 신당동
장충초등학교 다산동
효창공원앞역 6번 용문동
효창동주민센터 효창동
한강로동땡땡거리(은행나무길) 한강로동
효창공원앞역 5번 용문동
효창공원앞역 2번 효창동
용산구청 이태원1동
서빙고역 1번 서빙고동
경리단길북측 이태원2동
경리단길남측 이태원2동
이태원역 북측 이태원1동
남정초등학교 원효로1동
서울역 15번 청파동
배문고등학교 청파동
남영역 1번 원효로1동
숙대입구 청파동
용산세무서 한강로동
한강미주맨션아파트 이촌1동
열정도 원효로1동
삼

In [94]:
from elasticsearch import helpers
import json

vector = [item['_source']['title_vector'] for item in bulk_data]
print(len(vector[0]))

helpers.bulk(es, bulk_data)
print("inserting bulk is complete")

1024
inserting bulk is complete


## 검색

In [95]:
query_text = "이화"  # 검색할 텍스트
query_embedding = embeddings_model.embed_query(query_text)  # 쿼리를 벡터화

# Elasticsearch 검색 요청
search_query = {
    "query": {
        "script_score": {
            "query": {
                "match_all": {}  # 모든 문서에서 스코어 기반 필터링
            },
            "script": {
                "source": """
                    cosineSimilarity(params.query_vector, 'title_vector') + 
                    cosineSimilarity(params.query_vector, 'address_vector')
                """,
                "params": {
                    "query_vector": query_embedding
                }
            }
        }
    }
}


# 검색 요청 실행
response = es.search(index=es_index_name, body=search_query)

# 검색 결과 출력
print("벡터 검색 결과:")
for hit in response["hits"]["hits"]:
    print(hit["_source"])



벡터 검색 결과:
{'title': '이화달팽이길', 'address': '이화동', 'title_vector': [-0.03172757849097252, 0.010793512687087059, -0.023313041776418686, 0.0043387021869421005, -0.03872579708695412, 0.0021387541200965643, 0.04466956853866577, -0.040454596281051636, -0.015538881532847881, -0.012726929038763046, -0.023821962997317314, 0.04029069095849991, -0.0009561865590512753, -0.03516140952706337, 0.025136807933449745, -0.05055376514792442, 0.008708435110747814, 0.021021267399191856, 0.02934231236577034, 0.010731367394328117, -0.03294073045253754, -0.03294689580798149, -0.012409639544785023, 0.009153342805802822, -0.011544136330485344, 0.0017964337021112442, 0.023740556091070175, -0.03920717537403107, -0.002075024414807558, -0.005742769222706556, -0.02459261380136013, -0.006154181901365519, -0.002327926456928253, -0.048950497061014175, -0.031879961490631104, -0.03991160914301872, -0.03379659727215767, -0.0012833873042836785, -0.0769561305642128, 0.0589776337146759, 0.0027938582934439182, 0.0099803609773516